# Model Creation & Analysis

Two models will be created with Meta's Prophet. The first one will be a baseline with just the rental data while the other will use the employment of VA as an additional regressor. Afterwards, the models will be finetuned and analysis will be performed to examine the performances.

## Outline 
* Base model creation
* Model with additional employment regressor
* Midway comparison
* Fine tuning
* Statistical Analysis

In [34]:
from pyprojroot.here import here
import pandas as pd
import os

# Set the cwd to the project's root
os.chdir(here())
print(f"Current working directory: {os.getcwd()}")

Current working directory: /Users/amudh/Documents/DS4002/Project_2/MI1-2


In [35]:
import pandas as pd
# Create dataframe for combined data
combined_data = pd.read_csv('DATA/Charlottesville_Rent_Employment_Master.csv')

# Preview data
print( "COMBINED DATA")
combined_data.head(10)

COMBINED DATA


,ds,y,employment_count
0,2015-01-31,1030.293753,103503.0
1,2015-02-28,1033.140296,105522.0
2,2015-03-31,1036.707494,105700.0
3,2015-04-30,1057.235869,107064.0
4,2015-05-31,1060.458311,105949.0
5,2015-06-30,1066.297180,104874.0
6,2015-07-31,1063.034752,104150.0
7,2015-08-31,1071.748772,102822.0
8,2015-09-30,1080.462792,105301.0
9,2015-10-31,1102.797649,105007.0


First we have to account for COVID. Prophet has a "holiday" feature which we will use to account for the dip in our data. Through looking at our graphs and reflecting on our experience we decided on Jan - March 2020 with a lower window of 1 month and upper window of 3 months.

* `lower_window` : effect of the holiday that extends *before* the holiday begins
* `upper_window` : effect of the holiday that extends *after* the holiday ends

In [36]:
covid_dates = pd.date_range(start='2020-01-01', end='2020-03-31')
covid = pd.DataFrame({
  'holiday': 'covid',
  'ds': covid_dates,
  'lower_window': -30, # Lower window = effects that may have occured prior to covid ~ 1 month
  'upper_window': 90, # Upper window = effects that may have lasted past covid ~ 3 months
})


## Model 1: Base Model Creation

No NAN values should exist as the data was interpolated previously. Fitting the model will be limited to dates in 2024 so we can compare with the actual values in 2025.

In [37]:
from prophet import Prophet

# Take only the date and rental data from the combined data
rental_data = combined_data[['ds', 'y']].copy()
# Limit to 2024
rental_data = rental_data[rental_data['ds'] <= '2024-12-31']

m1 = Prophet(holidays=covid)
m1.fit(rental_data)

# Build the model excluding the year of 2025 
future = m1.make_future_dataframe(periods=12, freq='ME')
# Predict 2025 data
forecast1 = m1.predict(future)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast1[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

15:38:20 - cmdstanpy - INFO - Chain [1] start processing
15:38:20 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1831.227179,1812.020957,1848.925769
121,2025-02-28,1832.696562,1813.738096,1851.574925
122,2025-03-31,1847.027733,1827.369614,1866.764259
123,2025-04-30,1866.787131,1849.452432,1886.069416
124,2025-05-31,1874.536675,1854.271115,1892.929926
125,2025-06-30,1883.439654,1865.280862,1902.404523
126,2025-07-31,1892.294027,1874.313114,1911.398866
127,2025-08-31,1896.911332,1879.016798,1915.598729
128,2025-09-30,1905.019240,1886.204444,1924.810513
129,2025-10-31,1912.996144,1895.400068,1932.868760


## Model 2: Model with Unemployment Regressor


In [38]:
# Technically the same as the combined data set, but creating a copy for cleaner handling
rental_and_employment = combined_data[['ds', 'y', 'employment_count']].copy()
# Limit to end of 2024
rental_and_employment = rental_and_employment[rental_and_employment['ds'] <= '2024-12-31']

m2 = Prophet(holidays=covid)
# Registering the employment data as an additional regressor
m2.add_regressor('employment_count')
m2.fit(rental_and_employment)

# Build the model excluding the year of 2025 
future2 = m2.make_future_dataframe(periods=12, freq='ME')

# Prophet requires that regressor values be present in both fitting and prediction df
# Therefore the employment count will track to 2025 following real data
# First ensure dataframes are of the same type
combined_data['ds'] = pd.to_datetime(combined_data['ds'])
future2['ds'] = pd.to_datetime(future2['ds'])

# Merge data together to add 2025 employment count
future2 = future2.merge(
    combined_data[['ds', 'employment_count']],
    on='ds',
    how='left'
)
# Actual prediction
forecast2 = m2.predict(future2)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast2[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

15:38:22 - cmdstanpy - INFO - Chain [1] start processing
15:38:22 - cmdstanpy - INFO - Chain [1] done processing


,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1832.144667,1813.088194,1850.448105
121,2025-02-28,1832.468818,1813.462479,1851.759382
122,2025-03-31,1846.462016,1828.166893,1863.976323
123,2025-04-30,1865.385866,1847.182278,1883.948109
124,2025-05-31,1872.314633,1853.697036,1891.005582
125,2025-06-30,1881.708063,1863.304237,1902.030896
126,2025-07-31,1890.923337,1870.337984,1909.816606
127,2025-08-31,1895.872872,1876.399793,1915.115014
128,2025-09-30,1902.161829,1883.006035,1920.648343
129,2025-10-31,1909.878830,1890.908590,1929.210196


## Midway Comparison
Now that the rental data has been forecasted by the two models, we can do a preliminary comparison before any statistical tests.

In [39]:
# Pull actual values from the year of 2025
actual_values = combined_data[combined_data['ds'] >= '2025-01-01'][['ds', 'y']].copy()

# Pull the model performances 
f1 = forecast1[forecast1['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m1', 'yhat_lower': 'lower_m1', 'yhat_upper': 'upper_m1'})
f2 = forecast2[forecast2['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m2', 'yhat_lower': 'lower_m2', 'yhat_upper': 'upper_m2'})

# Merge all the data together for easy comparison
midway_comparison = actual_values.merge(f1, on='ds').merge(f2, on='ds')
# Check whether the actual value falls between the upper and lower ends of the model's prediction
midway_comparison['within_m1'] = midway_comparison['y'].between(midway_comparison['lower_m1'], midway_comparison['upper_m1'])
midway_comparison['within_m2'] = midway_comparison['y'].between(midway_comparison['lower_m2'], midway_comparison['upper_m2'])
# Confidence levels
midway_comparison['width_m1'] = midway_comparison['upper_m1'] - midway_comparison['lower_m1']
midway_comparison['width_m2'] = midway_comparison['upper_m2'] - midway_comparison['lower_m2']

print(midway_comparison[['ds', 'y', 'yhat_m1', 'within_m1', 'yhat_m2', 'within_m2']])
print(f"\nM1 coverage: {midway_comparison['within_m1'].mean():.0%}")
print(f"M2 coverage: {midway_comparison['within_m2'].mean():.0%}")
print(f"M1 avg interval width: {midway_comparison['width_m1'].mean():.2f}")
print(f"M2 avg interval width: {midway_comparison['width_m2'].mean():.2f}")

           ds            y      yhat_m1  within_m1      yhat_m2  within_m2
0  2025-01-31  1810.942452  1831.227179      False  1832.144667      False
1  2025-02-28  1853.575750  1832.696562      False  1832.468818      False
2  2025-03-31  1873.731004  1847.027733      False  1846.462016      False
3  2025-04-30  1873.613465  1866.787131       True  1865.385866       True
4  2025-05-31  1885.487821  1874.536675       True  1872.314633       True
5  2025-06-30  1895.184392  1883.439654       True  1881.708063       True
6  2025-07-31  1907.515917  1892.294027       True  1890.923337       True
7  2025-08-31  1898.018927  1896.911332       True  1895.872872       True
8  2025-09-30  1943.949582  1905.019240      False  1902.161829      False
9  2025-10-31  1952.670747  1912.996144      False  1909.878830      False
10 2025-11-30  1947.121214  1915.580941      False  1912.812333      False
11 2025-12-31  1908.264675  1921.073403       True  1919.339272       True

M1 coverage: 50%
M2 cove

## Finetuning 
The models did not perform as well as initially believed, but we will continue with finetuning to see if we can improve the performance.


In [40]:
from prophet.diagnostics import cross_validation, performance_metrics
import itertools
import numpy as np

# Paraneter grid 
param_grid = {
    'changepoint_prior_scale': [0.01, 0.05, 0.1], 
    'seasonality_prior_scale': [0.1, 1.0, 10.0],
    'changepoint_range': [0.8, 0.9, 0.95]
}

# Generate all combinations of params
all_params = [dict(zip(param_grid.keys(), v)) for v in itertools.product(*param_grid.values())]

# First finetune the base model
rmses_m1 = []  # RMSEs values for the parameters for model 1 

for params in all_params:
    m = Prophet(holidays=covid, **params)
    m.fit(rental_data)
    
    df_cv = cross_validation(m, initial='1825 days', period='180 days', horizon='365 days')
    df_p = performance_metrics(df_cv, rolling_window=1)
    rmses_m1.append(df_p['mae'].values[0])

# Find the best params
best_params_m1 = all_params[np.argmin(rmses_m1)]
print(f" Best parameters for base model: {best_params_m1}")



15:38:23 - cmdstanpy - INFO - Chain [1] start processing
15:38:23 - cmdstanpy - INFO - Chain [1] done processing
  0%|          | 0/8 [00:00<?, ?it/s]15:38:24 - cmdstanpy - INFO - Chain [1] start processing
15:38:25 - cmdstanpy - INFO - Chain [1] done processing
 12%|█▎        | 1/8 [00:02<00:17,  2.51s/it]15:38:26 - cmdstanpy - INFO - Chain [1] start processing
15:38:27 - cmdstanpy - INFO - Chain [1] done processing
 25%|██▌       | 2/8 [00:04<00:13,  2.25s/it]15:38:28 - cmdstanpy - INFO - Chain [1] start processing
15:38:29 - cmdstanpy - INFO - Chain [1] done processing
 38%|███▊      | 3/8 [00:05<00:09,  1.85s/it]15:38:29 - cmdstanpy - INFO - Chain [1] start processing
15:38:30 - cmdstanpy - INFO - Chain [1] done processing
 50%|█████     | 4/8 [00:07<00:07,  1.81s/it]15:38:31 - cmdstanpy - INFO - Chain [1] start processing
15:38:33 - cmdstanpy - INFO - Chain [1] done processing
 62%|██████▎   | 5/8 [00:10<00:06,  2.01s/it]15:38:33 - cmdstanpy - INFO - Chain [1] start processing
15:

 Best parameters for base model: {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 0.1, 'changepoint_range': 0.95}


In [41]:

# Finetune the second model
rmses_m2 = []  # RMSEs values for the parameters for model 2

for params in all_params:
    m = Prophet(holidays=covid, **params)
    m.add_regressor('employment_count')
    m.fit(rental_and_employment)
    
    df_cv = cross_validation(m, initial='1825 days', period='180 days', horizon='365 days')
    df_p = performance_metrics(df_cv, rolling_window=1)
    rmses_m2.append(df_p['mae'].values[0])

# Find the best params
best_params_m2 = all_params[np.argmin(rmses_m2)]
print(f" Best parameters for second model: {best_params_m2}")

15:45:33 - cmdstanpy - INFO - Chain [1] start processing
15:45:33 - cmdstanpy - INFO - Chain [1] done processing
  0%|          | 0/8 [00:00<?, ?it/s]15:45:33 - cmdstanpy - INFO - Chain [1] start processing
15:45:35 - cmdstanpy - INFO - Chain [1] done processing
 12%|█▎        | 1/8 [00:01<00:13,  1.96s/it]15:45:35 - cmdstanpy - INFO - Chain [1] start processing
15:45:36 - cmdstanpy - INFO - Chain [1] done processing
 25%|██▌       | 2/8 [00:03<00:11,  1.95s/it]15:45:37 - cmdstanpy - INFO - Chain [1] start processing
15:45:39 - cmdstanpy - INFO - Chain [1] done processing
 38%|███▊      | 3/8 [00:06<00:10,  2.03s/it]15:45:39 - cmdstanpy - INFO - Chain [1] start processing
15:45:41 - cmdstanpy - INFO - Chain [1] done processing
 50%|█████     | 4/8 [00:08<00:09,  2.25s/it]15:45:42 - cmdstanpy - INFO - Chain [1] start processing
15:45:43 - cmdstanpy - INFO - Chain [1] done processing
 62%|██████▎   | 5/8 [00:10<00:06,  2.18s/it]15:45:44 - cmdstanpy - INFO - Chain [1] start processing
15:

 Best parameters for second model: {'changepoint_prior_scale': 0.1, 'seasonality_prior_scale': 0.1, 'changepoint_range': 0.95}


Now we run fit the models again with their proper hyperparameters

In [42]:
m1_tuned = Prophet(holidays=covid, **best_params_m1)
m1_tuned.fit(rental_data)

m2_tuned = Prophet(holidays=covid, **best_params_m2)
m2_tuned.add_regressor('employment_count')
m2_tuned.fit(rental_and_employment)

15:52:31 - cmdstanpy - INFO - Chain [1] start processing
15:52:31 - cmdstanpy - INFO - Chain [1] done processing
15:52:32 - cmdstanpy - INFO - Chain [1] start processing
15:52:32 - cmdstanpy - INFO - Chain [1] done processing


And then we can run the forecasts a final time

### Model 1:

In [43]:
# Build the model excluding the year of 2025 
future = m1_tuned.make_future_dataframe(periods=12, freq='ME')
# Predict 2025 data
forecast1_tuned = m1_tuned.predict(future)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast1_tuned[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1799.232596,1783.223003,1816.220137
121,2025-02-28,1795.882540,1778.853675,1812.659201
122,2025-03-31,1805.019445,1789.158456,1821.348224
123,2025-04-30,1819.552596,1802.654557,1836.711010
124,2025-05-31,1821.494328,1805.385303,1839.607685
125,2025-06-30,1825.081887,1807.092141,1843.417716
126,2025-07-31,1828.528297,1811.286291,1844.432123
127,2025-08-31,1827.739006,1810.905862,1845.950709
128,2025-09-30,1830.869425,1813.161389,1849.834624
129,2025-10-31,1833.757800,1815.691863,1852.504960


### Model 2

In [44]:
future2 = m2_tuned.make_future_dataframe(periods=12, freq='ME')

# Prophet requires that regressor values be present in both fitting and prediction df
# Therefore the employment count will track to 2025 following real data
# First ensure dataframes are of the same type
combined_data['ds'] = pd.to_datetime(combined_data['ds'])
future2['ds'] = pd.to_datetime(future2['ds'])

# Merge data together to add 2025 employment count
future2 = future2.merge(
    combined_data[['ds', 'employment_count']],
    on='ds',
    how='left'
)
# Actual prediction
forecast2_tuned = m2_tuned.predict(future2)

# Create a data frame with yhat as the predicted value per row and columns for uncertainty levels
forecast2_tuned[['ds', 'yhat', 'yhat_lower', 'yhat_upper']].tail(12)

,ds,yhat,yhat_lower,yhat_upper
120,2025-01-31,1798.811733,1782.762870,1815.674447
121,2025-02-28,1796.236728,1780.160122,1812.749213
122,2025-03-31,1805.499225,1787.823231,1821.125744
123,2025-04-30,1820.504161,1803.332375,1837.044966
124,2025-05-31,1822.886656,1806.355611,1840.234517
125,2025-06-30,1826.132220,1808.983958,1842.757439
126,2025-07-31,1829.384670,1812.100180,1847.461123
127,2025-08-31,1828.420628,1810.122959,1845.933396
128,2025-09-30,1832.585334,1813.494676,1851.039471
129,2025-10-31,1835.577104,1816.079098,1854.298572


### Final Comparison

In [45]:
# Pull actual values from the year of 2025
actual_values = combined_data[combined_data['ds'] >= '2025-01-01'][['ds', 'y']].copy()

# Pull the model performances 
f1 = forecast1_tuned[forecast1_tuned['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m1', 'yhat_lower': 'lower_m1', 'yhat_upper': 'upper_m1'})
f2 = forecast2_tuned[forecast2_tuned['ds'] >= '2025-01-01'][['ds', 'yhat', 'yhat_lower', 'yhat_upper']].rename(
    columns={'yhat': 'yhat_m2', 'yhat_lower': 'lower_m2', 'yhat_upper': 'upper_m2'})

# Merge all the data together for easy comparison
final_comparison = actual_values.merge(f1, on='ds').merge(f2, on='ds')
# Check whether the actual value falls between the upper and lower ends of the model's prediction
final_comparison['within_m1'] = final_comparison['y'].between(final_comparison['lower_m1'], final_comparison['upper_m1'])
final_comparison['within_m2'] = final_comparison['y'].between(final_comparison['lower_m2'], final_comparison['upper_m2'])
# Confidence levels
final_comparison['width_m1'] = final_comparison['upper_m1'] - final_comparison['lower_m1']
final_comparison['width_m2'] = final_comparison['upper_m2'] - final_comparison['lower_m2']

print(final_comparison[['ds', 'y', 'yhat_m1', 'within_m1', 'yhat_m2', 'within_m2']])
print(f"\nM1 coverage: {final_comparison['within_m1'].mean():.0%}")
print(f"M2 coverage: {final_comparison['within_m2'].mean():.0%}")
print(f"M1 avg interval width: {final_comparison['width_m1'].mean():.2f}")
print(f"M2 avg interval width: {final_comparison['width_m2'].mean():.2f}")

           ds            y      yhat_m1  within_m1      yhat_m2  within_m2
0  2025-01-31  1810.942452  1799.232596       True  1798.811733       True
1  2025-02-28  1853.575750  1795.882540      False  1796.236728      False
2  2025-03-31  1873.731004  1805.019445      False  1805.499225      False
3  2025-04-30  1873.613465  1819.552596      False  1820.504161      False
4  2025-05-31  1885.487821  1821.494328      False  1822.886656      False
5  2025-06-30  1895.184392  1825.081887      False  1826.132220      False
6  2025-07-31  1907.515917  1828.528297      False  1829.384670      False
7  2025-08-31  1898.018927  1827.739006      False  1828.420628      False
8  2025-09-30  1943.949582  1830.869425      False  1832.585334      False
9  2025-10-31  1952.670747  1833.757800      False  1835.577104      False
10 2025-11-30  1947.121214  1831.712779      False  1833.273623      False
11 2025-12-31  1908.264675  1832.380896      False  1833.412242      False

M1 coverage: 8%
M2 cover

After multiple attempts for hyper parameter tuning it seems that the default models performed the best overall.

## Statistical Analysis

Though there was a very small difference between the performance of the two models, we will continue with the statistical test just to see if the performance difference is statistically significant.

In [46]:
from dieboldmariano import dm_test

# Using the midway values that were not tuned as these were the best
result = dm_test(
    midway_comparison['y'].values,
    midway_comparison['yhat_m1'].values,
    midway_comparison['yhat_m2'].values,
    h=1,          # forecast horizon = 12 months
    one_sided=False # two-sided for H0: M1 == M2
)

dm_stat, p_value = result

print("Diebold-Mariano Test")
print(f"DM Statistic: {dm_stat:.4f}")
print(f"P-value:      {p_value:.4f}")

if p_value < 0.05:
    print("Result: Reject H0. There is a tatistically significant difference between models")
    # The sign of the DM value determines which model performs better. M1 = first model M2 = second
    if dm_stat > 0:
        print("Direction: M2 (employment) outperforms M1 (base)")
    else:
        print("Direction: M1 (base) outperforms M2 (employment)")
else:
    print("Result: Fail to reject H0. There is no statistically significant difference between models")

Diebold-Mariano Test
DM Statistic: -2.6344
P-value:      0.0232
Result: Reject H0. There is a tatistically significant difference between models
Direction: M1 (base) outperforms M2 (employment)


After checking the statistical test, it seems like M1 performed better. The Diebold-Mariano test result is statistically significant (p=0.0232), and because our loss differential was calculated as M1−M2, the negative result confirms that Model 1 consistently produced smaller errors. The next cell will look at the mean loss to see how the models were performing. After running the cell it can be seen that M1 has smaller errors overall which explains the results of the DM test.

(Negative values → M1 has smaller error
Positive values → M2 has smaller error)

In [47]:
# Calculation of the mean loss difference
midway_comparison['e1'] = midway_comparison['y'] - midway_comparison['yhat_m1']
midway_comparison['e2'] = midway_comparison['y'] - midway_comparison['yhat_m2']
midway_comparison['diff'] = midway_comparison['e1']**2 - midway_comparison['e2']**2

print(midway_comparison[['ds', 'diff']])
print("Mean loss diff:", midway_comparison['diff'].mean())

           ds        diff
0  2025-01-31  -38.063773
1  2025-02-28   -9.562073
2  2025-03-31  -30.533035
3  2025-04-30  -21.094555
4  2025-05-31  -53.605286
5  2025-06-30  -43.672575
6  2025-07-31  -43.607800
7  2025-08-31   -3.378783
8  2025-09-30 -230.644806
9  2025-10-31 -257.073972
10 2025-11-30 -182.310442
11 2025-12-31   41.416817
Mean loss diff: -72.67752354254101


Let's check the MAE and perform two more tests to really check what this means for the two models.

In [48]:
from sklearn.metrics import mean_absolute_error

# Mean absolute error calculation
mae1 = mean_absolute_error(midway_comparison['y'], midway_comparison['yhat_m1'])
mae2 = mean_absolute_error(midway_comparison['y'], midway_comparison['yhat_m2'])

print(f"Model 1 MAE: {mae1}")
print(f"Model 2 MAE: {mae2}")

Model 1 MAE: 19.722736333256005
Model 2 MAE: 21.09641952792657


In [ ]:
from scipy.stats import ttest_rel
from scipy.stats import wilcoxon

# Absolute error calculation
e1 = abs(midway_comparison['y'] - midway_comparison['yhat_m1'])
e2 = abs(midway_comparison['y'] - midway_comparison['yhat_m2'])

t_stat, p_t = ttest_rel(e1, e2) # Checks the difference of averages
w_stat, p_w = wilcoxon(e1, e2) # Checks the difference of medians

print(f"Paired t-test (abs error): t={t_stat:.6f}, p={p_t:.6g}")
print(f"Wilcoxon signed-rank (abs error): w={w_stat:.6f}, p={p_w:.6g}")

Paired t-test (abs error): t=-3.530727, p=0.00470882
Wilcoxon signed-rank (abs error): w=8.000000, p=0.012207


## Conclusion

Overall even though the two models seemed to have similar values, the base model proved to be better and our tests proved the statistical differences.

**Diebold-Mariano**: With a DM value of -2.6344 and p-value of p-value of 0.0232, we reject the null hypothesis and can confirm that Model 1 is the mathematically the better forecaster.

**T-Test / Wilcoxon Test**: Confirms the "gap" between the two models is not due to noise (difference in the averages and medians).

**MAE vs. Mean Loss**: Model 1 has a lower MAE (19.72) compared to Model 2 (21.10). The mean loss difference was -72.68 which further confirms the lower error rates.